# RAAI-SGC 2.0 — Main project: Colab T4 LLM inference and guarded QLoRA

**Same canonical project folder:** `MyDrive/RAAI_SGC_2_COMPLETE_PROJECT`.
This notebook does not download or create a second project. It reads the dataset and code from the
main project folder, works from a temporary Colab copy, and saves only outputs back to Drive.

**Run:** Runtime → Change runtime type → **T4 GPU**. The GPU and approved-label requirements
are checked before any training; the corpus is unreviewed and does not guarantee factual accuracy.

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, shutil, os, json, time

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/RAAI_SGC_2_COMPLETE_PROJECT')
assert (DRIVE_ROOT/'data/processed/master.csv').exists(), f'Cannot find dataset in {DRIVE_ROOT}'
assert (DRIVE_ROOT/'src/hybrid.py').exists(), 'Missing retrieval code'
WORK = Path('/content/RAAI_SGC_2_COMPLETE_PROJECT')
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(DRIVE_ROOT, WORK, ignore=shutil.ignore_patterns('.venv','__pycache__','.pytest_cache','models/checkpoints'))
os.chdir(WORK)
sys.path.insert(0,str(WORK))
print('Main project copied for runtime:', WORK)
print('Corpus path:',WORK/'data/processed/master.csv')

## Install model dependencies
PyTorch comes with Colab. We use an instruction-tuned 1.5B model to stay within typical T4 memory,
although allocation may fail if the GPU is unavailable or memory is exhausted. Pin versions for reproducibility after first successful run.

In [ ]:
%pip -q install 'transformers>=4.48,<5' 'accelerate>=1,<2' 'scikit-learn>=1.3,<2' 'peft>=0.15,<1' 'bitsandbytes>=0.45' 'datasets>=3,<5'
import torch, transformers
print('Torch:',torch.__version__,'Transformers:',transformers.__version__)
print('CUDA available:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable Runtime > Change runtime type > T4 GPU before loading the LLM')
print('GPU:',torch.cuda.get_device_name(0))

## Retrieve evidence from the project's dataset
For the research study we must keep retrieval settings identical across comparison arms.

In [ ]:
from src.hybrid import load_data, HybridRetriever
from src.grounding import evidence_messages, has_evidence
sources,edges,drafts=load_data(WORK/'data/processed/master.csv')
retriever=HybridRetriever(sources,edges,backend='tfidf')
QUESTION='What is Kubernetes?'
hits,diagnostic=retriever.search(QUESTION,mode='hybrid',top_k=3,alpha=0.25)
print('Sources:',len(sources),'Edges:',sum(map(len,edges.values())),'Draft QA excluded:',drafts)
print('Graph activated:',diagnostic['graph_activated'])
for n,hit in enumerate(hits,1):
    print(n,hit['source_title'],'TFIDF:',hit['vector_score'],'Graph:',hit['graph_score'],hit['source_url'])
assert hits and hits[0]['source_title']=='Kubernetes'

## Pretrained model baseline (no fine-tuning)
Model output may contain mistakes. Source list represents retrieved context, **not independently verified answer citations**.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
BASE_MODEL='Qwen/Qwen2.5-1.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL)
model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,torch_dtype=torch.float16,device_map='auto')
generator=pipeline('text-generation',model=model,tokenizer=tokenizer)
assert has_evidence(hits), 'Insufficient retrieval score; do not generate an answer'
messages=evidence_messages(QUESTION,hits)
out=generator(messages,max_new_tokens=180,do_sample=False,return_full_text=False)
from src.llm import extract_generated
answer=extract_generated(out)
print('QUESTION:',QUESTION,'\n\nGENERATED ANSWER:\n',answer)
print('\nRETRIEVED SOURCES:')
for n,h in enumerate(hits,1): print(f'[{n}]',h['source_url'])

## Save baseline inference record to your **same** Drive project
This is a model run record, not a validated accuracy measurement.

In [ ]:
from datetime import datetime,timezone
record={'timestamp_utc':datetime.now(timezone.utc).isoformat(),
        'base_model':BASE_MODEL,'question':QUESTION,'retrieval_backend':'tfidf',
        'retrieval_mode':'hybrid','top_k':3,'alpha':0.25,
        'source_ids':[h['source_id'] for h in hits],
        'answer':answer,'generation':{'do_sample':False,'max_new_tokens':180},
        'torch_version':torch.__version__,'transformers_version':transformers.__version__,
        'warning':'Not reviewed for factual correctness or citation faithfulness.'}
target=DRIVE_ROOT/'reports/generated/colab_llm_baseline.json'
target.parent.mkdir(parents=True,exist_ok=True)
target.write_text(json.dumps(record,indent=2),encoding='utf-8')
print('Saved:',target)

## Controlled pretrained LLM demonstrations across retrieval methods
The **same pretrained model, question, token budget, and corpus** are used for three methods. This records real answers for human review; no automatic correctness metric is claimed.

In [ ]:
# Execute AFTER the pretrained generator is loaded above.
from datetime import datetime, timezone
modes=('vector','unweighted','hybrid')
comparisons=[]
for mode in modes:
    evidence,diagnostics=retriever.search(QUESTION,mode=mode,top_k=3,alpha=0.25)
    if not has_evidence(evidence):
        generated='INSUFFICIENT_EVIDENCE'
    else:
        output=generator(evidence_messages(QUESTION,evidence),max_new_tokens=180,do_sample=False,return_full_text=False)
        generated=extract_generated(output)
    comparisons.append(dict(evaluation_id=f'baseline-{mode}',question=QUESTION,retrieval_mode=mode,
        answer=generated,source_ids=[h['source_id'] for h in evidence],graph_activated=diagnostics['graph_activated'],
        model=BASE_MODEL,timestamp_utc=datetime.now(timezone.utc).isoformat()))
    print(mode,':',generated[:700])
path=DRIVE_ROOT/'reports/generated/colab_three_way_predictions.json'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(json.dumps(comparisons,indent=2,ensure_ascii=False),encoding='utf8')
print('SAVED actual model outputs to',path)


## Generate a blank human-review rubric
The script leaves scoring columns blank. Expert reviewers—not the model or this notebook—must score correctness, groundedness, completeness and citation accuracy.

In [ ]:
from evaluation.prepare_answer_review import create
review=DRIVE_ROOT/'reports/generated/colab_answer_review_BLANK.csv'
create(path,review)
print('Saved review template:',review)


## Optional QLoRA — approval gate, **disabled by default**
Do not change the gate to bypass it. After real human review, export approved TRAIN-only
examples into `data/training/sft_train.jsonl`, then rerun the notebook. **No QLoRA performance
claim can be made without approved held-out evaluation.**

In [ ]:
from training.train_qlora_colab import load_approved
TRAIN_FILE=DRIVE_ROOT/'data/training/sft_train.jsonl'
try:
    approved=load_approved(TRAIN_FILE)
    print('Approved training records:',len(approved))
    TRAINING_ALLOWED=True
except (ValueError,RuntimeError,FileNotFoundError) as exc:
    TRAINING_ALLOWED=False
    print('Training intentionally BLOCKED:',exc)
print('Training enabled?',TRAINING_ALLOWED)

In [ ]:
# Explicit opt-in only after a non-empty, approved, TRAIN-only file exists.
RUN_QLORA=TRAINING_ALLOWED # automatic ONLY after genuine human-approved examples exist
if RUN_QLORA:
    if not TRAINING_ALLOWED: raise RuntimeError('Training gate failed; do not bypass review.')
    import subprocess
    local_train=WORK/'data/training/sft_train.jsonl'
    shutil.copyfile(TRAIN_FILE,local_train)
    del generator,model,tokenizer
    import gc; gc.collect();torch.cuda.empty_cache()
    cmd=[sys.executable,'-m','training.train_qlora_colab','--model',BASE_MODEL,
         '--data',str(local_train),'--output',str(WORK/'models/checkpoints/colab_qlora'),
         '--epochs','1','--max-steps','30']
    cmd += ['--approved-export',str(WORK/'data/review/APPROVED_QA_ONLY.csv'),'--master',str(WORK/'data/processed/master.csv')]
    subprocess.run(cmd,check=True,cwd=WORK)
    target=DRIVE_ROOT/'models/checkpoints/colab_qlora'
    target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copytree(WORK/'models/checkpoints/colab_qlora',target,dirs_exist_ok=True)
    print('Adapter saved to same Drive project:',target)
else:
    print('QLoRA was not executed; inference baseline only.')

## Save runtime and checkpoint evidence
This cell records GPU and package versions, plus whether an actual adapter exists. A missing adapter is correctly reported as missing.

In [ ]:
from pathlib import Path
adapter_dir=DRIVE_ROOT/'models/checkpoints/colab_qlora'
run_audit={
 'timestamp_utc':datetime.now(timezone.utc).isoformat(),
 'gpu_name':torch.cuda.get_device_name(0),
 'pytorch':torch.__version__,'transformers':transformers.__version__,
 'base_model':BASE_MODEL,
 'qlora_adapter_saved':(adapter_dir/'adapter_config.json').exists(),
 'approved_training_records':len(approved) if TRAINING_ALLOWED else 0,
 'note':'No human answer-quality ratings are inferred.'}
run_file=DRIVE_ROOT/'reports/generated/colab_gpu_execution_evidence.json'
run_file.write_text(json.dumps(run_audit,indent=2),encoding='utf8')
print(json.dumps(run_audit,indent=2))


## Research next steps

1. Approve independently checked training questions and sources; do not use the 290 drafts as approved labels.
2. Review the independent test questions with domain experts.
3. Execute `evaluation/compare_retrievers.py` after the independent gold set is approved.
4. Evaluate model answers with a transparent human faithfulness/correctness rubric.
5. Keep GPU hardware, model revision, prompt settings, and evaluation questions consistent across ablations.

GPU inference and training cells are intentionally **not pre-executed** here. Run them on Colab T4.